In [0]:
# should handle late records
# Merge Schema  ===issue
# Record-level deduplication  === issue
# Late-arriving data (using business date)
# Upserts (MERGE, not append)✔# Idempotency (safe reruns)
# Schema stability (controlled)
# Basic audit columns

In [0]:
from pyspark.sql.functions import col, row_number, current_timestamp
from pyspark.sql.window import Window
from delta.tables import DeltaTable

dfsilverraw=spark.read.format("delta").table("deltacatalog.deltaschema.bronze_sales")
dfsilverraw.display()

In [0]:

dfsilverSch = dfsilverraw.withColumn("ProductKey",col("ProductKey").cast("double"))
dfsilverSch.display()


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,col

windowspec=Window.partitionBy("ID","OrderNumber").orderBy(col("ingestion_Timestamp").desc())
dfsilverdup=dfsilverSch.withColumn("rank",row_number().over(windowspec))\
                .filter(col("rank") == 1)\
                .drop("rank")  

dfsilverdup.display()          


In [0]:
dfsilverdup.createOrReplaceTempView("silverdup")

In [0]:
%sql
select * from silverdup

In [0]:
%sql
MERGE INTO deltacatalog.deltaschema.silver_sales t
USING silverdup s
ON ON t.OrderNumber = s.OrderNumber AND t.ID = s.ID
WHEN MATCHED THEN
  UPDATE SET
    t.date = s.date,
    t.ID = s.ID,
    t.ProductKey = s.ProductKey,
    t.OrderQuantity = s.OrderQuantity,
    t.ingestion_Timestamp = s.ingestion_Timestamp,
    t.ingestion_date = s.ingestion_date,
    t.source = s.source,
    t.file_name = s.file_name

WHEN NOT MATCHED THEN
  INSERT (
    date,
    ID,
    OrderNumber,
    ProductKey,
    OrderQuantity,
    ingestion_Timestamp,
    ingestion_date,
    source,
    file_name
  )
  VALUES (
    s.date,
    s.ID,
    s.OrderNumber,
    s.ProductKey,
    s.OrderQuantity,
    s.ingestion_Timestamp,
    s.ingestion_date,
    s.source,
    s.file_name
  );

In [0]:
silver_sales=spark.sql("select * from deltacatalog.deltaschema.silver_sales")
silver_sales.display()